# 04 · Machine‑Learning Forecasts – Random Forest & XGBoost
We now move from classical time‑series baselines to **supervised ML** that can learn simultaneously from:
* temporal lags (t‑1, t‑3, t‑12),
* spatial lag (last month’s neighbour average),
* static ward attributes (IMD score, population),
* month‑of‑year seasonality encoded as categorical.

Algorithms compared:
* **RandomForestRegressor** (scikit‑learn)
* **XGBRegressor** (XGBoost) – if the package is installed.

Evaluation horizon = calendar year **2024** (12 months) on *all* wards.

In [1]:
# Install required packages
%pip install pandas numpy scikit-learn pathlib xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pathlib, pandas as pd, numpy as np, warnings, datetime as dt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
ROOT = pathlib.Path('..').resolve()
FP = ROOT / 'data_cache/processed/features.parquet'
df = pd.read_parquet(FP)
df.head(50)

,Month,WD24CD,WD24NM,burglaries,lag_1,lag_3,lag_12,lag_spatial_avg,imd_score,population
0,2023-04-01,E05000946,Molyneux,1,0.0,0.0,0.0,0.000000,NaN,NaN
1,2022-10-01,E05000968,Oxton,1,0.0,0.0,0.0,0.000000,NaN,NaN
2,2022-06-01,E05000969,Pensby and Thingwall,1,0.0,0.0,0.0,0.000000,NaN,NaN
3,2018-02-01,E05001305,Blakenall,1,0.0,0.0,0.0,0.000000,NaN,NaN
4,2024-12-01,E05001314,Pleck,1,0.0,0.0,0.0,0.000000,NaN,NaN
5,2018-05-01,E05001316,St Matthew's,1,0.0,0.0,0.0,0.000000,NaN,NaN
6,2023-11-01,E05002219,Prittlewell,1,0.0,0.0,0.0,0.000000,NaN,NaN
7,2012-02-01,E05002229,Aveley and Uplands,1,0.0,0.0,0.0,5.400000,NaN,NaN
8,2012-03-01,E05002229,Aveley and Uplands,2,1.0,0.0,0.0,9.000000,NaN,NaN
9,2012-04-01,E05002229,Aveley and Uplands,1,2.0,0.0,0.0,6.400000,NaN,NaN


## Train / test split (temporal)

In [3]:
train = df[df['Month'] < '2024-01']
test  = df[df['Month'] >= '2024-01']

y_train = train['burglaries']
y_test  = test['burglaries']

### Feature matrix

In [4]:
num_features   = ['lag_1','lag_3','lag_12','lag_spatial_avg','imd_score','population']
cat_features   = ['month']

# create 'month' integer 1‑12
for frame in (train, test):
    frame['month'] = frame['Month'].dt.month

X_train = train[num_features + cat_features]
X_test  = test[num_features + cat_features]

## Helper: build preprocessing + regressor pipeline

In [5]:
def make_pipe(model):
    cat_ohe = OneHotEncoder(handle_unknown='ignore')
    pre = ColumnTransformer([
        ('num', 'passthrough', num_features),
        ('cat', cat_ohe, cat_features)
    ])
    return Pipeline([('pre', pre), ('model', model)])

## 1 · Random Forest

In [6]:
rf = RandomForestRegressor(n_estimators=400, max_depth=None, random_state=0, n_jobs=-1)
pipe_rf = make_pipe(rf)
pipe_rf.fit(X_train, y_train)
pred_rf = pipe_rf.predict(X_test)
mae_rf = mean_absolute_error(y_test, pred_rf)
mae_rf

2.7010785824015793

## 2 · XGBoost

In [7]:
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=0,
        n_jobs=-1)
    pipe_xgb = make_pipe(xgb)
    pipe_xgb.fit(X_train, y_train)
    pred_xgb = pipe_xgb.predict(X_test)
    mae_xgb = mean_absolute_error(y_test, pred_xgb)
except ImportError:
    mae_xgb = np.nan
    print('xgboost not installed - skipping')

## Results

In [8]:
results = pd.Series({
    'Random Forest MAE': mae_rf,
    'XGBoost MAE': mae_xgb
}).to_frame('MAE')
results

,MAE
Random Forest MAE,2.701079
XGBoost MAE,2.613019


### Per‑ward MAE (Random Forest)

In [9]:
test = test.copy()
test['pred_rf'] = pred_rf
ward_mae = (test.groupby('WD24CD').apply(lambda g: mean_absolute_error(g['burglaries'], g['pred_rf'])))
ward_mae.describe()

count    715.000000
mean       2.568333
std        1.175695
min        0.000000
25%        1.970268
50%        2.477692
75%        3.083956
max       16.120893
dtype: float64